In [ ]:
import requests
from datetime import datetime
from langchain.document_loaders import DirectoryLoader, TextLoader, WebBaseLoader
from langchain.text_splitter import CharacterTextSplitter, RecursiveCharacterTextSplitter
from dotenv import load_dotenv, find_dotenv 
from langchain.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.prompts import ChatPromptTemplate
from langchain.chat_models import ChatOpenAI
from langchain.schema.runnable import RunnablePassthrough
from langchain.schema.output_parser import StrOutputParser

from download_cards import download_model_cards
from utils import *

from retriever import Retriever
from generator import Generator

_ = load_dotenv(find_dotenv())

# Indexing 

In [ ]:
repo_url_evidently = "https://github.com/evidentlyai/evidently"
repo_name_evidently = repo_url_evidently.rstrip('/').split('/')[-1]
extract_dir_evidently = f"./{repo_name_evidently}"

repo_url_dh = "https://scc-digitalhub.github.io/docs/"

download_repo = True
if download_repo:
    download_and_extract_repo(repo_url_evidently, extract_dir_evidently)

#py_files = get_py_files(extract_dir_evidently)

In [ ]:
# Step 1.1: load documents
def split_into_chunks(repo_folder):
    #loader = DirectoryLoader('model_cards/', glob="**/*.md", loader_cls=TextLoader)
    loader = DirectoryLoader(f"{repo_folder}/", glob="**/*.py", loader_cls=TextLoader)
    documents = loader.load()
    # Step 1.2: split documents into chunks
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    chunks = text_splitter.split_documents(documents)
    return chunks

def split_into_chunks_documentation(web_url):
    loader = WebBaseLoader(web_url)
    documents = loader.load()
    # Step 1.2: split documents into chunks
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    chunks = text_splitter.split_documents(documents)
    return chunks

chunks_evidently = split_into_chunks(extract_dir_evidently)
chunks_dh = split_into_chunks_documentation(repo_url_dh)

In [ ]:
model_name = "sentence-transformers/all-mpnet-base-v2"
model_kwargs = {'device': 'cpu'}
encode_kwargs = {'normalize_embeddings': False}
embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)
# Step 1.3: encode chunks into vectors and store in a vector database
vectordb_evidently = FAISS.from_documents(chunks_evidently, embeddings)
vectordb_evidently.save_local("vectorstore_evidently.db")

vectordb_dh = FAISS.from_documents(chunks_dh, embeddings)
vectordb_dh.save_local("vectorstore_dh.db")

# Merge vector stores
vectordb_evidently.merge_from(vectordb_dh)


# Retrieval

In [ ]:
def pretty_print_docs(docs):
    print(
        f"\n{'-' * 100}\n".join(
            [
                f"Document {i + 1}:\n\n{d.page_content}\nMetadata: {d.metadata}"
                for i, d in enumerate(docs)
            ]
        )
    )

In [ ]:
# Step 2: Retrieval: retrieve the Top k chunks most relevant to the question based on semantic similarity.
retriever = vectordb_evidently.as_retriever()
docs = retriever.invoke(query)
pretty_print_docs(docs)

## Reranker

In [ ]:
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_community.document_compressors import FlashrankRerank    
from flashrank import Ranker 

ranker = Ranker(model_name="ms-marco-MiniLM-L-12-v2")
compressor = FlashrankRerank(model="ms-marco-MiniLM-L-12-v2", top_n=3)
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=retriever
)
query = "Please generate a new python script that detects data drift for tabular data using your context?"
compressed_docs = compression_retriever.invoke(query)
pretty_print_docs(compressed_docs)

# Generation

In [ ]:
generator = Generator(model="gpt-4o")
template = generator.format_prompt_codegen(
    system_prompt_path="prompts/system_prompt_codegen.txt", 
    user_prompt_path="prompts/user_prompt_codegen.txt")
template

## Prompt Engineering


In [ ]:
!python -m spacy download en_core_web_sm

In [ ]:
import spacy
import json

# Load English language model
nlp = spacy.load("en_core_web_sm")

# Sample text with meta data candidates
text = """In a blog post titled ‘The Top 10 Tech Trends of 2024,’ 
John Doe discusses the rise of artificial intelligence and machine learning 
in various industries. The article mentions companies like Google and Microsoft 
as pioneers in AI research. Additionally, it highlights emerging technologies 
such as natural language processing and computer vision."""

# Process the text with spaCy
doc = nlp(text)

# Extract named entities and their labels
meta_data = [{"text": ent.text, "label": ent.label_} for ent in doc.ents]

# Convert meta data to JSON format
meta_data_json = json.dumps(meta_data)

print(meta_data_json)

In [ ]:
prompt = ChatPromptTemplate.from_template(template)
print(prompt)

In [ ]:
# Step 3: Generation: input the original question and the retrieved chunks together into LLM to generate the final answer.
llm = ChatOpenAI(model_name="gpt-4o", temperature=0.5)

rag_chain = (
    {"context": retriever,  "question": RunnablePassthrough()} 
    | prompt 
    | llm
    | StrOutputParser() 
)

query = "Please generate a new python script that detects data drift for tabular data using your context?"
result = rag_chain.invoke(query)
current_time = datetime.now()
with open(f"results/generated_{current_time}.py", "w") as file:
    file.write(result)

In [ ]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_community.document_compressors.rankllm_rerank import RankLLMRerank
    
#Option 1
#compressor = RankLLMRerank(top_n=3, model="gpt", gpt_model="gpt-4o-mini")
#compression_retriever = ContextualCompressionRetriever(
#    base_compressor=compressor, base_retriever=retriever
#)

#Option 2
#torch.cuda.empty_cache()
#compressor = RankLLMRerank(top_n=3, model="rank_zephyr")
#compression_retriever = ContextualCompressionRetriever(
#    base_compressor=compressor, base_retriever=retriever
#)
#del compressor


# RAG Evaluation

In [ ]:
# Import necessary libraries
from ragas.testset.generator import TestsetGenerator
from ragas.testset.evolutions import simple, reasoning, multi_context
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# Instantiate the models
generator_llm = ChatOpenAI(model="gpt-4o-mini")
critic_llm = ChatOpenAI(model="gpt-4o-mini")
embeddings = OpenAIEmbeddings()

# Create the TestsetGenerator
generator = TestsetGenerator.from_langchain(
    generator_llm,
    critic_llm,
    embeddings
)

# Call the generator
testset = generator.generate_with_langchain_docs(
data_transformed, 
test_size=20, 
distributions={ 
simple: 0.5, 
reasoning: 0.25, 
multi_context: 0.25}
)